# ReLoG: Review Locally, Recommend Globally

Federated training and evaluation of ReLoG. Set `DATASET`, `SEEDS` and `CONFIGS` in the configuration cell
and run all cells.

- `CONFIGS = [dict(mode='none')]` trains ReLoG without differential privacy (warm-user and few-shot results).
- `dict(mode='central', z=...)` and `dict(mode='local', z=...)` train the user-level DP variants DP-A and DP-B.
  The privacy budget is computed with `dp-accounting` at `DELTA`.
- `ATTACK_EVERY > 0` runs the membership-inference and review-leakage attacks of an honest-but-curious server
  on the updates it observes.

Ranking is computed in fp32 over the full catalogue; the rank of the target is the number of items with a
strictly higher score.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import copy
import random
import math

In [ ]:
import os
import dp_accounting
from dp_accounting import pld as dp_pld, rdp as dp_rdp

DATASET = 'movies'   # movies | cd | electronics
SEEDS = [0, 1, 2, 3, 4]
ROUNDS = 100

# mode 'none': ReLoG without DP
# mode 'central': DP-A, Gaussian noise on the securely aggregated sum
# mode 'local': DP-B, each client adds Gaussian noise to its own update
CONFIGS = [dict(mode='none')]   # e.g. dict(mode='central', z=1.0), dict(mode='local', z=5.0)
CLIP = {'movies': 0.41, 'cd': 0.39, 'electronics': 0.40}[DATASET]
Q_SAMPLE, DELTA = 0.05, 1e-5
ATTACK_EVERY = 0   # > 0: run the server-side attacks at round 1 and every ATTACK_EVERY rounds

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

NAME = {'movies': 'Movies_and_TV', 'cd': 'CD_Vinyl', 'electronics': 'electronics'}[DATASET]


def find_parquet(fname):
    for root in ['preprocessing', '../preprocessing']:
        p = os.path.join(root, fname)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(fname)


df_sampled = pd.read_parquet(find_parquet(f'{NAME}_review.parquet'))
df_meta_aligned = pd.read_parquet(find_parquet(f'{NAME}_meta.parquet'))

NUM_USERS = df_sampled['user_id_int'].nunique()
NUM_ITEMS = df_sampled['item_id_int'].nunique()
INTERACTIONS = len(df_sampled)
SPARSITY = (1 - (INTERACTIONS / (NUM_USERS * NUM_ITEMS))) * 100

print(f"Dataset: {NAME}")
print(f"Users: {NUM_USERS}, Items: {NUM_ITEMS}, Interactions: {INTERACTIONS}")
print(f"Sparsity: {SPARSITY:.2f}%")

## Calculating Embeddings

User reviews and item metadata records are encoded
using SBERT (`all-MiniLM-L6-v2`) into dense 384-dimensional vectors. These
embeddings serve as the input for the two towers: the reviews describe the user, and the
metadata describes the item.

In [ ]:
sbert = SentenceTransformer('all-MiniLM-L6-v2', device=device)

df_sampled['summary']    = df_sampled['summary'].fillna('')
df_sampled['reviewText'] = df_sampled['reviewText'].fillna('')
df_sampled['full_text']  = df_sampled['summary'] + ". " + df_sampled['reviewText']

print("Calculating embeddings review...")
review_embeddings = sbert.encode(
    df_sampled['full_text'].tolist(),
    batch_size=64, show_progress_bar=True, convert_to_numpy=True
)
review_emb_map = {i: review_embeddings[i] for i in range(len(df_sampled))}

print("Calculating embeddings metadata item...")
meta_embeddings = sbert.encode(
    df_meta_aligned['meta_text'].tolist(),
    batch_size=64, show_progress_bar=True, convert_to_numpy=True
)
item_meta_tensor = torch.tensor(meta_embeddings, dtype=torch.float32)
print(f"Item bank: {item_meta_tensor.shape}")

item_meta_tensor_gpu = item_meta_tensor.to(device)

## Architecture Definition

The architecture is *two-tower*:

- **`UserTower`** and **`ItemTower`** project the SBERT embeddings into a
  common, L2-normalized latent space.
- **`LocalScoreFunction`** (`client_mlp`) is a small *local* MLP, specific
  to each user: it concatenates the user and item representations and produces the
  relevance score.

The model is trained using **BPR loss**, which ranks positive items above
negative ones.

In [ ]:
class UserTower(nn.Module):

    def __init__(self, input_dim=384, hidden_dim=512, output_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)


class ItemTower(nn.Module):

    def __init__(self, input_dim=384, hidden_dim=512, output_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, text_emb):
        return F.normalize(self.net(text_emb), dim=-1)


class LocalScoreFunction(nn.Module):

    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, user_emb, item_emb):
        if user_emb.shape[0] != item_emb.shape[0]:
            if user_emb.shape[0] == 1:
                user_emb = user_emb.expand(item_emb.shape[0], -1)
            else:
                raise RuntimeError(f"Shape mismatch: {user_emb.shape} vs {item_emb.shape}")
        x = torch.cat([user_emb, item_emb], dim=-1)
        return self.net(x)


class TwoTowerRecommender(nn.Module):

    def __init__(self, input_dim=384, hidden_dim=512, output_dim=256, inference_temperature=0.07):
        super().__init__()
        self.item_tower = ItemTower(input_dim, hidden_dim, output_dim)
        self.user_tower = UserTower(input_dim, hidden_dim, output_dim)
        self.client_mlp = LocalScoreFunction(input_dim=output_dim * 2, hidden_dim=128)
        self.inference_temperature = inference_temperature

    def get_user_repr(self, review_embeddings):
        return self.user_tower(review_embeddings).mean(dim=0, keepdim=True)

    def get_item_repr(self, item_meta_embeddings):
        return self.item_tower(item_meta_embeddings)

    def training_score(self, user_repr, item_reprs):
        raw_scores = self.client_mlp(user_repr, item_reprs).squeeze(-1)
        return raw_scores / self.inference_temperature

    def score(self, user_repr, item_reprs):
        return self.training_score(user_repr, item_reprs)


def bpr_loss(pos_scores, neg_scores):
    diff = pos_scores.unsqueeze(1) - neg_scores.unsqueeze(0)
    return -F.logsigmoid(diff).mean()

## Data Utilities

`get_client_data` constructs a dataset for a single user using a
time-based split: the oldest interactions for training, the second-to-last for
validation, and the most recent for testing.

In [ ]:
_CLIENT_CACHE = {}

def get_client_data(user_id, df, emb_map, mode="train"):
    key = (id(df), int(user_id), mode)
    if key in _CLIENT_CACHE:
        return _CLIENT_CACHE[key]

    user_df = df[df['user_id_int'] == user_id].sort_values('timestamp')
    if len(user_df) < 3:
        _CLIENT_CACHE[key] = (None, None)
        return None, None
    indices = user_df.index.tolist()
    train_idx = indices[:-2]
    val_idx   = indices[-2]
    test_idx  = indices[-1]
    X_train = torch.tensor(
        np.array([emb_map[i] for i in train_idx]),
        dtype=torch.float32
    )
    train_item_ids = user_df.loc[train_idx, 'item_id_int'].tolist()
    if mode == "val":
        target_id = int(user_df.loc[val_idx, 'item_id_int'])
    elif mode == "test":
        target_id = int(user_df.loc[test_idx, 'item_id_int'])
    else:
        target_id = None
    _CLIENT_CACHE[key] = ((X_train, train_item_ids), target_id)
    return _CLIENT_CACHE[key]

## Hard Negative Sampling

Instead of random negatives, the model selects the uninteracted items to which the
model assigns the highest score: more informative “hard” negatives, which
make the BPR loss more effective.

In [ ]:
def sample_hard_negatives(user_repr, pos_set, all_metas, local_model,
                          num_neg, num_candidates, device, num_total_items):
    all_ids = np.arange(num_total_items)
    pos_arr = np.array(list(pos_set), dtype=np.int64)
    mask = np.ones(num_total_items, dtype=bool)
    mask[pos_arr] = False
    eligible = all_ids[mask]

    if len(eligible) < num_neg:
        return eligible.tolist()

    n_cands = min(num_candidates, len(eligible))
    candidate_ids = np.random.choice(eligible, size=n_cands, replace=False)

    with torch.no_grad():
        cand_ids_tensor = torch.tensor(candidate_ids, device=device)
        cand_metas = all_metas[cand_ids_tensor]
        cand_reprs = local_model.get_item_repr(cand_metas)
        cand_scores = local_model.training_score(user_repr.detach(), cand_reprs)

    top_k = min(num_neg, len(candidate_ids))
    top_indices = torch.topk(cand_scores, top_k).indices.cpu().numpy()
    return candidate_ids[top_indices].tolist()

## Local Client Training

Each client (user) starts with the global weights, trains the two towers and its
own score function locally, and returns the tower weights to be aggregated. The
learning rate follows a linear warmup followed by cosine annealing.

In [ ]:
def get_lr(base_lr, current_step, warmup_steps, total_steps):
    if current_step < warmup_steps:
        return base_lr * (current_step + 1) / warmup_steps
    progress = (current_step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))

def train_client(user_id, global_state_dict, X_train_reviews, train_item_ids,
                 all_metas_gpu,
                 device, client_states,
                 lr=0.001, epochs=5, num_neg=10, use_hard_negatives=True,
                 lr_warmup_steps=10, current_step=0, total_steps=100):

    local_model = TwoTowerRecommender().to(device)
    local_model.load_state_dict(global_state_dict, strict=False)

    user_local_data = client_states.get(user_id, None)
    if user_local_data is not None:
        local_model.client_mlp.load_state_dict(user_local_data)

    effective_lr = get_lr(lr, current_step, lr_warmup_steps, total_steps)
    optimizer = torch.optim.Adam(local_model.parameters(), lr=effective_lr)
    local_model.train()

    X_train = X_train_reviews.to(device)
    pos_set = set(train_item_ids)
    num_total_items = all_metas_gpu.shape[0]

    pos_tensor = torch.tensor(train_item_ids, device=device)
    pos_metas  = all_metas_gpu[pos_tensor]

    loss = None
    for _ in range(epochs):
        optimizer.zero_grad()
        user_repr = local_model.get_user_repr(X_train)

        pos_reprs = local_model.get_item_repr(pos_metas)

        if use_hard_negatives:
            neg_ids = sample_hard_negatives(
                user_repr, pos_set, all_metas_gpu, local_model,
                num_neg=len(train_item_ids) * num_neg,
                num_candidates=2000, device=device,
                num_total_items=num_total_items
            )
        else:
            all_ids = np.arange(num_total_items)
            mask = np.ones(num_total_items, dtype=bool)
            mask[np.array(list(pos_set))] = False
            neg_ids = np.random.choice(
                all_ids[mask],
                size=len(train_item_ids) * num_neg,
                replace=True
            ).tolist()

        neg_tensor = torch.tensor(neg_ids, device=device)
        neg_metas  = all_metas_gpu[neg_tensor]
        neg_reprs  = local_model.get_item_repr(neg_metas)

        pos_scores = local_model.training_score(user_repr, pos_reprs)
        neg_scores = local_model.training_score(user_repr, neg_reprs)

        loss = bpr_loss(pos_scores, neg_scores)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(local_model.parameters(), 1.0)
        optimizer.step()

    client_states[user_id] = local_model.client_mlp.state_dict()

    shared_state = {k: v.cpu() for k, v in local_model.state_dict().items()
                    if 'client_mlp' not in k}

    return shared_state, loss.item(), len(train_item_ids)

## Federated Aggregation (FedAvg + momentum)

The weights of the selected clients' towers are averaged in proportion to
the number of interactions for each. A *momentum* term stabilizes
the update of the global model across successive rounds. The score function
remains local to each user and is not aggregated.

In [ ]:
def weighted_fedavg_momentum(global_model, local_weights_list, local_sizes,
                             momentum_buffer, beta=0.9):
    total_samples = sum(local_sizes)
    global_dict   = global_model.state_dict()
    keys_to_agg   = [k for k in global_dict if 'client_mlp' not in k]

    if momentum_buffer is None:
        momentum_buffer = {k: torch.zeros_like(global_dict[k]) for k in keys_to_agg}

    with torch.no_grad():
        for key in keys_to_agg:
            layer_avg = torch.zeros_like(global_dict[key])
            for i, w in enumerate(local_weights_list):
                layer_avg.add_(w[key].to(layer_avg.device),
                               alpha=local_sizes[i] / total_samples)

            delta = layer_avg - global_dict[key]
            momentum_buffer[key].mul_(beta).add_(delta, alpha=1 - beta)
            global_dict[key].add_(momentum_buffer[key])

    global_model.load_state_dict(global_dict, strict=False)
    return momentum_buffer

In [ ]:
def tower_delta(local_state, round_state):
    return {k: local_state[k].to(round_state[k].device) - round_state[k] for k in local_state}


def flat_norm(delta):
    return float(torch.sqrt(sum((v.float() ** 2).sum() for v in delta.values())))


def dp_fedavg_momentum(global_model, round_state, local_weights_list, mode, clip, z,
                       expected_clients, momentum_buffer, beta, noise_gen, observe=None):
    keys = list(round_state.keys())
    total = {k: torch.zeros_like(round_state[k]) for k in keys}
    norms = []
    for i, w in enumerate(local_weights_list):
        d = tower_delta({k: w[k] for k in keys}, round_state)
        n = flat_norm(d)
        norms.append(n)
        scale = min(1.0, clip / (n + 1e-12))
        released = {}
        for k in keys:
            upd = d[k] * scale
            if mode == 'local' and z > 0:
                upd = upd + torch.randn(upd.shape, generator=noise_gen, device=upd.device) * (z * clip)
            total[k] += upd
            released[k] = upd
        if observe is not None and mode != 'central':
            observe(i, released)
    if mode == 'central' and z > 0:
        for k in keys:
            total[k] += torch.randn(total[k].shape, generator=noise_gen, device=total[k].device) * (z * clip)
    avg = {k: total[k] / expected_clients for k in keys}
    if observe is not None and mode == 'central':
        observe(None, avg)

    if momentum_buffer is None:
        momentum_buffer = {k: torch.zeros_like(round_state[k]) for k in keys}
    global_dict = global_model.state_dict()
    with torch.no_grad():
        for k in keys:
            momentum_buffer[k].mul_(beta).add_(avg[k], alpha=1 - beta)
            global_dict[k].add_(momentum_buffer[k])
    global_model.load_state_dict(global_dict, strict=False)
    return momentum_buffer, norms


def privacy_accounting(mode, z, q, rounds, max_participations, delta=1e-5):
    if mode == 'none' or z <= 0:
        return float('inf'), float('inf')
    if mode == 'central':
        event = dp_accounting.SelfComposedDpEvent(
            dp_accounting.PoissonSampledDpEvent(q, dp_accounting.GaussianDpEvent(z)), rounds)
    else:
        event = dp_accounting.SelfComposedDpEvent(dp_accounting.GaussianDpEvent(z), max(1, max_participations))
    pld = dp_pld.PLDAccountant(); pld.compose(event)
    rdp = dp_rdp.RdpAccountant(); rdp.compose(event)
    return pld.get_epsilon(delta), rdp.get_epsilon(delta)

In [ ]:
from scipy.stats import rankdata

ATTACK_RNG = np.random.default_rng(0)
USER_ITEMS = df_sampled.groupby('user_id_int')['item_id_int'].apply(lambda s: np.unique(s.values)).to_dict()
N_NONMEMBERS, N_OTHER_USERS = 1000, 20


# membership inference (Yuan et al., WWW 2023): AUC of members vs. sampled non-members
def mia_auc(scores, members, user_items):
    mask = np.ones(len(scores), dtype=bool); mask[user_items] = False
    non = ATTACK_RNG.choice(np.flatnonzero(mask), size=N_NONMEMBERS, replace=False)
    s = np.concatenate([scores[members], scores[non]])
    r = rankdata(s)
    n_p = len(members)
    return float((r[:n_p].sum() - n_p * (n_p + 1) / 2) / (n_p * N_NONMEMBERS))


class ServerAttacker:
    def __init__(self, train_users):
        self.probe = TwoTowerRecommender().to(device).eval()
        self.train_users = np.asarray(train_users)
        self.mean_review = {}
        self.res = {'auc': [], 'sim_own': [], 'sim_other': [], 'round': []}

    def user_mean_review(self, uid):
        if uid not in self.mean_review:
            X, _ = get_client_data(uid, df_sampled, review_emb_map)[0]
            self.mean_review[uid] = F.normalize(X.mean(0), dim=0).to(device)
        return self.mean_review[uid]

    @torch.no_grad()
    def item_reprs(self, tower_state):
        self.probe.load_state_dict(tower_state, strict=False)
        return torch.cat([self.probe.get_item_repr(item_meta_tensor_gpu[i:i + 8192])
                          for i in range(0, item_meta_tensor_gpu.shape[0], 8192)])

    @torch.no_grad()
    def start_round(self, round_num, tower_state):
        self.round_num, self.tower_state = round_num, tower_state
        self.base = self.item_reprs(tower_state)

    @torch.no_grad()
    def attack(self, update, victims):
        moved = self.item_reprs({k: self.tower_state[k] + update[k].to(self.tower_state[k].device)
                                 for k in self.tower_state})
        scores = (moved - self.base).norm(dim=1).cpu().numpy()
        w = update['user_tower.net.0.weight'].float()
        # review leakage: top right singular vector of the first user-tower layer update
        v1 = torch.linalg.svd(w, full_matrices=False)[2][0]
        for uid in victims:
            uid = int(uid)
            (_, train_ids), _ = get_client_data(uid, df_sampled, review_emb_map)
            self.res['auc'].append(mia_auc(scores, np.asarray(train_ids), USER_ITEMS[uid]))
            self.res['sim_own'].append(abs(float(v1 @ self.user_mean_review(uid))))
            others = ATTACK_RNG.choice(self.train_users[self.train_users != uid], N_OTHER_USERS, replace=False)
            self.res['sim_other'].append(float(np.mean([abs(float(v1 @ self.user_mean_review(int(o))))
                                                        for o in others])))
            self.res['round'].append(self.round_num)

    def summary(self):
        r = {k: np.asarray(v) for k, v in self.res.items()}
        if not len(r['auc']):
            return {}
        by_round = {int(t): float(r['auc'][r['round'] == t].mean()) for t in np.unique(r['round'])}
        return {'n_obs': int(len(r['auc'])), 'auc': float(r['auc'].mean()), 'auc_std': float(r['auc'].std()),
                'auc_by_round': by_round, 'sim_own': float(r['sim_own'].mean()),
                'sim_other': float(r['sim_other'].mean())}

## Evaluation (warm users)

**HR@k** and **NDCG@k** metrics using the leave-one-out protocol: for each user,
a brief local fine-tuning is performed, and then we check whether the target item is among the
first *k* recommended items.

During this fine-tuning only the score function is updated, so the tower outputs
are computed once per user; the target is ranked against all non-interacted items
(its rank is the number of items scoring strictly higher, in fp32). Evaluation
randomness uses a dedicated generator (`EVAL_RNG`), so training does not depend
on how often we validate.

In [ ]:
EVAL_RNG  = np.random.default_rng(0)
EVAL_TIES = {"users_with_ties": 0}


def rank_target(target_score, neg_scores, k):
    n_higher = int((neg_scores > target_score).sum().item())
    if bool((neg_scores == target_score).any().item()):
        EVAL_TIES["users_with_ties"] += 1
    if n_higher < k:
        return 1, 1.0 / np.log2(n_higher + 2)
    return 0, 0.0


def reset_mlp(mlp):
    for m in mlp.modules():
        if isinstance(m, nn.Linear):
            m.reset_parameters()


def finetune_and_rank(local_model, user_repr, all_item_embs, pos_ids, target_id,
                      num_items, k, lr, epochs, device):
    batch_pos = pos_ids if len(pos_ids) < 32 else EVAL_RNG.choice(pos_ids, 32, replace=False).tolist()
    pos_mask = np.zeros(num_items, dtype=bool)
    pos_mask[pos_ids] = True
    eligible_neg = np.flatnonzero(~pos_mask)
    pos_reprs = all_item_embs[torch.tensor(batch_pos, device=device)]

    optimizer = torch.optim.Adam(local_model.client_mlp.parameters(), lr=lr)
    local_model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        neg_idx   = EVAL_RNG.choice(eligible_neg, size=min(len(batch_pos), len(eligible_neg)), replace=False)
        neg_reprs = all_item_embs[torch.tensor(neg_idx, device=device)]
        loss = bpr_loss(
            local_model.training_score(user_repr, pos_reprs),
            local_model.training_score(user_repr, neg_reprs)
        )
        loss.backward()
        optimizer.step()

    local_model.eval()
    with torch.no_grad():
        neg_mask = torch.ones(num_items, dtype=torch.bool, device=device)
        neg_mask[torch.tensor(pos_ids, device=device)] = False
        neg_mask[target_id] = False
        target_score = local_model.score(user_repr, all_item_embs[target_id].unsqueeze(0)).item()
        neg_scores   = local_model.score(user_repr, all_item_embs[neg_mask])
    return rank_target(target_score, neg_scores, k)


def build_eval_model(global_model, all_metas_gpu, device):
    local_model = TwoTowerRecommender(
        inference_temperature=global_model.inference_temperature).to(device)
    local_model.load_state_dict(global_model.state_dict(), strict=False)
    local_model.eval()
    with torch.no_grad():
        all_item_embs = torch.cat([
            local_model.get_item_repr(all_metas_gpu[i:i + 4096])
            for i in range(0, all_metas_gpu.shape[0], 4096)
        ], dim=0)
    return local_model, all_item_embs


def evaluate_top_k(global_model, eval_users, df, emb_map,
                   all_metas_gpu,
                   client_states, k=10, device='cuda', mode="test",
                   eval_fraction=1.0):
    if eval_fraction < 1.0:
        n_sample   = max(1, int(len(eval_users) * eval_fraction))
        eval_users = EVAL_RNG.choice(eval_users, n_sample, replace=False)

    num_items = all_metas_gpu.shape[0]
    local_model, all_item_embs = build_eval_model(global_model, all_metas_gpu, device)
    hits, ndcgs, count = 0, 0, 0

    for user_id in tqdm(eval_users, desc=f"Evaluating ({mode})"):
        train_data, target_id = get_client_data(user_id, df, emb_map, mode=mode)
        if train_data is None:
            continue
        X_train, train_ids = train_data

        user_local_data = client_states.get(user_id, None)
        if user_local_data is not None:
            local_model.client_mlp.load_state_dict(user_local_data)
        else:
            reset_mlp(local_model.client_mlp)

        with torch.no_grad():
            user_repr = local_model.get_user_repr(X_train.to(device))

        hit, ndcg = finetune_and_rank(local_model, user_repr, all_item_embs, train_ids,
                                      target_id, num_items, k, lr=0.005, epochs=3, device=device)
        hits += hit
        ndcgs += ndcg
        count += 1

    if count == 0:
        return 0.0, 0.0
    return hits / count, ndcgs / count

## Few-shot evaluation (unseen users)

For users not seen during training, we measure the model’s adaptability: we
use the first *k* interactions (1-, 2-, or 3-shot, or all in “full” mode)
to fine-tune only the score function, and we evaluate the prediction
of the next interaction.

In [ ]:
_FEWSHOT_CACHE = {}

def get_fewshot_data(user_id, df, emb_map, num_shots):
    key = (id(df), int(user_id), num_shots)
    if key in _FEWSHOT_CACHE:
        return _FEWSHOT_CACHE[key]

    user_df = df[df['user_id_int'] == user_id].sort_values('timestamp')

    if num_shots is None:
        if len(user_df) < 2:
            _FEWSHOT_CACHE[key] = (None, None, None)
            return None, None, None
        indices       = user_df.index.tolist()
        shot_idx      = indices[:-2]
        target_idx    = indices[-1]
    else:
        min_required = num_shots + 1
        if len(user_df) < min_required:
            _FEWSHOT_CACHE[key] = (None, None, None)
            return None, None, None
        indices    = user_df.index.tolist()
        shot_idx   = indices[:num_shots]
        target_idx = indices[num_shots]

    X_shots = torch.tensor(
        np.array([emb_map[i] for i in shot_idx]),
        dtype=torch.float32
    )
    shot_item_ids = user_df.loc[shot_idx, 'item_id_int'].tolist()
    target_id     = int(user_df.loc[target_idx, 'item_id_int'])

    _FEWSHOT_CACHE[key] = (X_shots, shot_item_ids, target_id)
    return _FEWSHOT_CACHE[key]


def evaluate_fewshot(global_model, unseen_users, df, emb_map,
                     all_metas_gpu, num_shots,
                     k=10, device='cuda', finetune_epochs=5, lr=0.01):
    label = f"{num_shots}-shot" if num_shots is not None else "full"
    num_items = all_metas_gpu.shape[0]
    local_model, all_item_embs = build_eval_model(global_model, all_metas_gpu, device)
    hits, ndcgs, count = 0, 0, 0

    for user_id in tqdm(unseen_users, desc=f"Few-shot eval ({label})", leave=False):
        X_shots, shot_item_ids, target_id = get_fewshot_data(user_id, df, emb_map, num_shots)
        if X_shots is None:
            continue

        reset_mlp(local_model.client_mlp)
        with torch.no_grad():
            user_repr = local_model.get_user_repr(X_shots.to(device))

        hit, ndcg = finetune_and_rank(local_model, user_repr, all_item_embs, shot_item_ids,
                                      target_id, num_items, k, lr=lr, epochs=finetune_epochs,
                                      device=device)
        hits += hit
        ndcgs += ndcg
        count += 1

    if count == 0:
        return 0.0, 0.0
    print(f"  [{label}] evaluated users: {count}/{len(unseen_users)}")
    return hits / count, ndcgs / count

## User Split

Users are divided into **warm** (seen during training) and **unseen** (used
only for few-shot evaluation), in a way that is reproducible given the same seed.

In [ ]:
def split_users(df, unseen_ratio=0.2, seed=42):
    rng = np.random.RandomState(seed)
    all_users = df['user_id_int'].unique()
    rng.shuffle(all_users)
    n_total = len(all_users)
    n_unseen  = int(n_total * unseen_ratio)
    unseen_users  = all_users[:n_unseen]
    train_users = all_users[n_unseen:]
    return train_users, unseen_users

## Experiment Definition

`run_experiment` orchestrates a complete run for a given seed: user splitting,
federated training for a fixed number of rounds with periodic evaluation,
selection of the best model on the validation set, and final testing on warm and unseen
users.

In [ ]:
def run_experiment(seed, cfg):
    mode, z = cfg['mode'], cfg.get('z', 0.0)
    print(f"\n===== {DATASET} | {mode} z={z} C={CLIP} | seed {seed} =====")
    set_seed(seed)
    EVAL_RNG.bit_generator.state = np.random.default_rng(seed + 10_000).bit_generator.state
    noise_gen = torch.Generator(device=device); noise_gen.manual_seed(seed + 20_000)
    ATTACK_RNG.bit_generator.state = np.random.default_rng(seed + 30_000).bit_generator.state

    train_users, unseen_users = split_users(df_sampled, unseen_ratio=0.2, seed=seed)
    print(f"Train users: {len(train_users)}  Unseen users: {len(unseen_users)}")

    LR                    = 0.0005
    LOCAL_EPOCHS          = 3
    NUM_NEG_TRAIN         = 10
    USE_HARD_NEG          = True
    CLIENTS_PER_ROUND     = round(0.05 * len(train_users))
    GLOBAL_ROUNDS         = ROUNDS
    EVAL_EVERY            = min(10, ROUNDS)
    INFERENCE_TEMPERATURE = 0.07
    FEDAVG_MOMENTUM       = 0.9
    K                     = 20
    LR_WARMUP_STEPS       = 10
    expected_clients      = Q_SAMPLE * len(train_users)

    client_states   = {user_id: None for user_id in train_users}
    participations  = {}
    best_val_hr, best_round = -1.0, None
    best_state = best_client_states = None
    momentum_buffer = None
    all_norms = []

    global_model = TwoTowerRecommender(inference_temperature=INFERENCE_TEMPERATURE).to(device)
    attacker = ServerAttacker(train_users) if ATTACK_EVERY > 0 else None

    for round_num in range(1, GLOBAL_ROUNDS + 1):
        local_weights, local_sizes, local_losses, local_users = [], [], [], []
        round_state_dict = global_model.state_dict()
        tower_state = {k: v.clone() for k, v in round_state_dict.items() if 'client_mlp' not in k}
        attack_now = attacker is not None and (round_num == 1 or round_num % ATTACK_EVERY == 0)
        observe = None
        if attack_now:
            attacker.start_round(round_num, tower_state)
            observe = lambda i, upd: attacker.attack(upd, local_users if i is None else [local_users[i]])

        # fixed 5% sample of the clients without DP, Poisson sampling with DP
        if mode == 'none':
            selected = np.random.choice(train_users, CLIENTS_PER_ROUND, replace=False)
        else:
            selected = train_users[np.random.random(len(train_users)) < Q_SAMPLE]
        for user_id in selected:
            train_data, _ = get_client_data(user_id, df_sampled, review_emb_map)
            if train_data is None:
                continue
            X_train, train_item_ids = train_data
            w, loss, n = train_client(
                user_id, round_state_dict, X_train, train_item_ids, item_meta_tensor_gpu, device,
                client_states, lr=LR, epochs=LOCAL_EPOCHS, num_neg=NUM_NEG_TRAIN,
                use_hard_negatives=USE_HARD_NEG, lr_warmup_steps=LR_WARMUP_STEPS,
                current_step=round_num - 1, total_steps=GLOBAL_ROUNDS)
            local_weights.append(w); local_sizes.append(n); local_losses.append(loss); local_users.append(user_id)
            participations[int(user_id)] = participations.get(int(user_id), 0) + 1

        if mode == 'none':
            deltas = [tower_delta({k: w[k] for k in tower_state}, tower_state) for w in local_weights]
            all_norms += [flat_norm(d) for d in deltas]
            if observe is not None:
                for i, d in enumerate(deltas):
                    observe(i, d)
            del deltas
            if local_weights:
                momentum_buffer = weighted_fedavg_momentum(
                    global_model, local_weights, local_sizes, momentum_buffer, beta=FEDAVG_MOMENTUM)
        else:
            momentum_buffer, norms = dp_fedavg_momentum(
                global_model, tower_state, local_weights, mode, CLIP, z, expected_clients,
                momentum_buffer, FEDAVG_MOMENTUM, noise_gen, observe=observe)
            all_norms += norms

        avg_loss = sum(local_losses) / max(1, len(local_losses))
        msg = f"{round_num:<6} | n={len(local_weights):<4} | {avg_loss:<8.4f}"
        if round_num % EVAL_EVERY == 0:
            val_hr, val_ndcg = evaluate_top_k(
                global_model, train_users, df_sampled, review_emb_map, item_meta_tensor_gpu,
                client_states, k=K, device=device, mode="val")
            if val_hr > best_val_hr:
                best_val_hr, best_round = val_hr, round_num
                best_state = copy.deepcopy(global_model.state_dict())
                best_client_states = copy.deepcopy(client_states)
                msg += "  <- Best"
            msg += f" | val HR {val_hr:.4f} NDCG {val_ndcg:.4f}"
        print(msg, flush=True)

    norms = np.array(all_norms)
    norm_q = {p: float(np.percentile(norms, p)) for p in (10, 25, 50, 75, 90)} if len(norms) else {}
    print("update norms (percentiles):", {p: round(v, 4) for p, v in norm_q.items()})
    if mode != 'none':
        print(f"clipped fraction: {(norms > CLIP).mean():.3f}")
    max_part = max(participations.values()) if participations else 0
    eps_pld, eps_rdp = privacy_accounting(mode, z, Q_SAMPLE, GLOBAL_ROUNDS, max_part, DELTA)
    print(f"privacy: eps(PLD)={eps_pld:.3f} eps(RDP)={eps_rdp:.3f} delta={DELTA} "
          f"(max participations per user: {max_part})")

    global_model.load_state_dict(best_state)
    warm_hr, warm_ndcg = evaluate_top_k(
        global_model, train_users, df_sampled, review_emb_map, item_meta_tensor_gpu,
        best_client_states, k=K, device=device, mode="test")
    print(f"best round {best_round} | Warm HR@{K}: {warm_hr:.4f} | NDCG@{K}: {warm_ndcg:.4f}")
    fewshot_results = {}
    for num_shots in [1, 2, 3, None]:
        label = f"{num_shots}-shot" if num_shots is not None else "full"
        hr, ndcg = evaluate_fewshot(
            global_model, unseen_users, df_sampled, review_emb_map, item_meta_tensor_gpu,
            num_shots=num_shots, k=K, device=device, finetune_epochs=5, lr=0.005)
        fewshot_results[label] = (hr, ndcg)
        print(f"  {label:<8}  HR@{K}: {hr:.4f}  |  NDCG@{K}: {ndcg:.4f}")

    out = {'dataset': DATASET, 'mode': mode, 'z': z, 'clip': CLIP, 'q': Q_SAMPLE, 'delta': DELTA, 'seed': seed,
           'eps_pld': eps_pld, 'eps_rdp': eps_rdp, 'max_participations': max_part, 'best_round': best_round,
           'val_hr': best_val_hr, 'warm': [warm_hr, warm_ndcg], 'fewshot': fewshot_results, 'norm_q': norm_q}
    if attacker is not None:
        out['attack'] = attacker.summary()
        print("attack:", {k: v for k, v in out['attack'].items() if k != 'auc_by_round'})
        print("attack AUC by round:", {k: round(v, 3) for k, v in out['attack']['auc_by_round'].items()})
    return out

## Multi-seed Execution and Aggregated Results

The experiment is repeated using multiple seeds; the results are aggregated as
**mean ± standard deviation** for each scenario (warm and few-shot).

In [ ]:
all_results = []
for cfg in CONFIGS:
    res = [run_experiment(s, cfg) for s in SEEDS]
    all_results += res
    print(f"\n=== {DATASET} | {cfg} | {len(SEEDS)} seed(s) | eps(PLD)={res[0]['eps_pld']:.3f} ===")
    rows = [('warm', [r['warm'] for r in res])] + \
           [(l, [r['fewshot'][l] for r in res]) for l in ['1-shot', '2-shot', '3-shot', 'full']]
    for name, vals in rows:
        hr = np.array([v[0] for v in vals]); nd = np.array([v[1] for v in vals])
        print(f"{name:<8} | HR@20 {hr.mean():.4f} ± {hr.std():.4f} | NDCG@20 {nd.mean():.4f} ± {nd.std():.4f}")